**Descripción:**

Ejecutar pruebas integrales del pipeline para verificar la idempotencia, consistencia e integridad de los datos entre Bronze, Silver y Gold.

In [0]:
WITH duplicados AS (
    SELECT
        ROW_NUMBER() OVER (
            PARTITION BY sensor_id, timestamp
            ORDER BY timestamp
        ) AS rn
    FROM ecomotive.bronze.iot_sensors_raw
),

conteos AS (
    SELECT
        (SELECT COUNT(*)
         FROM ecomotive.bronze.iot_sensors_raw) AS total_bronze,

        (SELECT COUNT(*)
         FROM duplicados
         WHERE rn > 1) AS duplicados,

        (SELECT COUNT(*)
         FROM ecomotive.silver.validos_iot_sensors) AS validos,

        (SELECT COUNT(*)
         FROM ecomotive.silver.rechazos_iot_sensor) AS rechazados
)

SELECT
    total_bronze,
    duplicados,
    total_bronze - duplicados AS registros_unicos,
    validos,
    rechazados,
    validos + rechazados AS total_silver,
    (total_bronze - duplicados) - (validos + rechazados)
        AS diferencia
FROM conteos;

total_bronze,duplicados,registros_unicos,validos,rechazados,total_silver,diferencia
103,35,68,52,16,68,0
